In [1]:
import os
import cv2
import json
# Define input and output directories
image_folder = "/path/to/datasets/KITTI/training/image_2"
annotation_folder = "/path/to/datasets/KITTI/training/label_2"

output_train = "/path/to/datasets/KITTI/train/"
output_val = "/path/to/datasets/KITTI/val/"

# Ensure output folder exists
os.makedirs(output_train, exist_ok=True)
os.makedirs(output_val, exist_ok=True)

In [2]:
# Class mapping based on provided information
class_mapping = {
    "Car": 0,
    "Van": 1,
    "Truck": 2,
    "Pedestrian": 3,
    "Person_sitting": 4,
    "Cyclist": 5,
    "Tram": 6 # ,
    # "Misc": 8,
    # "Don'tCare": 9 # Ignoring Misc and Don'tCare for fine-tuning
}

train_val_split = json.load(open("../../our_datasets/splits/Kitti_train_val_split.json", "r"))

In [3]:
# Function to convert bounding box to YOLO format
def convert_to_yolo(class_id, x_min, y_min, x_max, y_max, img_width, img_height):
    x_center = ((x_min + x_max) / 2) / img_width
    y_center = ((y_min + y_max) / 2) / img_height
    width = (x_max - x_min) / img_width
    height = (y_max - y_min) / img_height
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n"

In [4]:
# train split

output_folder = output_train + "labels/"
output_folder_imgs = output_train + "images/"
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_folder_imgs, exist_ok=True)
# Process each annotation file
for annotation_file in os.listdir(annotation_folder):
    if annotation_file.endswith(".txt") and annotation_file.split(".")[0] in train_val_split["train_imgs"]:
        # Get corresponding image file
        image_name = os.path.splitext(annotation_file)[0] + ".png"  
        image_path = os.path.join(image_folder, image_name)
        annotation_path = os.path.join(annotation_folder, annotation_file)
        output_path = os.path.join(output_folder, annotation_file)

        # Check if the image exists
        if os.path.exists(image_path):
            # Copy image to output folder
            os.system(f"cp {image_path} {output_folder_imgs}")
            # Read image dimensions
            img = cv2.imread(image_path)
            img_height, img_width, _ = img.shape

            with open(annotation_path, "r") as infile, open(output_path, "w") as outfile:
                for line in infile:
                    parts = line.split()
                    class_name = parts[0]

                    # Check if class exists in mapping
                    if class_name in class_mapping:
                        class_id = class_mapping[class_name]
                        x_min, y_min, x_max, y_max = map(float, parts[4:8])
                        yolo_line = convert_to_yolo(class_id, x_min, y_min, x_max, y_max, img_width, img_height)
                        outfile.write(yolo_line)


In [5]:
# val split

output_folder = output_val + "labels/"
output_folder_imgs = output_val + "images/"
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_folder_imgs, exist_ok=True)
# Process each annotation file
for annotation_file in os.listdir(annotation_folder):
    if annotation_file.endswith(".txt") and annotation_file.split(".")[0] in train_val_split["val_imgs"]:
        # Get corresponding image file
        image_name = os.path.splitext(annotation_file)[0] + ".png"  
        image_path = os.path.join(image_folder, image_name)
        annotation_path = os.path.join(annotation_folder, annotation_file)
        output_path = os.path.join(output_folder, annotation_file)

        # Check if the image exists
        if os.path.exists(image_path):
            # Copy image to output folder
            os.system(f"cp {image_path} {output_folder_imgs}")
            # Read image dimensions
            img = cv2.imread(image_path)
            img_height, img_width, _ = img.shape

            with open(annotation_path, "r") as infile, open(output_path, "w") as outfile:
                for line in infile:
                    parts = line.split()
                    class_name = parts[0]

                    # Check if class exists in mapping
                    if class_name in class_mapping:
                        class_id = class_mapping[class_name]
                        x_min, y_min, x_max, y_max = map(float, parts[4:8])
                        yolo_line = convert_to_yolo(class_id, x_min, y_min, x_max, y_max, img_width, img_height)
                        outfile.write(yolo_line)
